In [1]:
import asyncio
from src.RAG.Strategies_RAG_Inference.query_expander import QueryExpander

async def test():
    expander = QueryExpander(model_name="qwen2.5:7b")

    query = "provide details on car loan"
    expanded = await expander.expand_query(query)

    return expanded  # ✅ return instead of just printing

In [2]:
query_batch = await test()
print(query_batch)

[2026-04-18 15:51:25,753] [INFO] TextGuardrails initialized successfully.
[2026-04-18 15:51:25,756] [INFO] QueryExpander active with qwen2.5:7b (Ollama native).
[2026-04-18 15:51:25,771] [INFO] apply completed in 0.01s
[2026-04-18 15:51:30,800] [INFO] apply completed in 0.01s
[2026-04-18 15:51:30,806] [INFO] apply completed in 0.01s
[2026-04-18 15:51:30,806] [INFO] Expanded into 2 queries.
[2026-04-18 15:51:30,816] [INFO] expand_query completed in 5.05s
['discuss the terms and conditions for an automobile financing plan', 'detail the specifics of a vehicle loan product']


In [3]:
import re

def normalize_section(section: str) -> str:
    if not section:
        return "default"

    section = section.lower().strip()
    section = re.sub(r"[^\w\s]", "", section)
    section = re.sub(r"\s+", " ", section)

    return section

In [4]:
import sys
from typing import Set, List

from sqlalchemy import select, func

from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("section_service")
current_logger.set(logger)


class SectionService:
    """
    Service to extract UNIQUE normalized section names
    from ChunkModel JSONB metadata.
    """

    @track_performance
    async def get_normalized_sections(self) -> List[str]:
        """
        Fetch unique section_name values from JSONB metadata,
        normalize them, and return sorted list.
        """

        try:
            sections: Set[str] = set()

            async for session in get_session():

                # ✅ DB-level DISTINCT extraction (fast)
                stmt = select(
                    func.distinct(
                        ChunkModel.chunk_metadata["section_name"].astext
                    )
                ).where(
                    ChunkModel.chunk_metadata["section_name"].isnot(None)
                )

                result = await session.execute(stmt)

                for row in result.scalars().all():
                    if not row:
                        continue

                    normalized = normalize_section(str(row).strip())
                    sections.add(normalized)

                break  # single session usage

            logger.info(f"Extracted {len(sections)} normalized sections.")
            return sorted(sections)

        except Exception as e:
            logger.error("Failed to extract normalized section names.")
            raise CustomException(e, sys)

In [5]:
sections_names = await SectionService().get_normalized_sections()

2026-04-18 15:52:12,435 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-18 15:52:12,435 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 15:52:12,587 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-18 15:52:12,587 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 15:52:12,660 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-18 15:52:12,663 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 15:52:12,690 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 15:52:12,750 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 15:52:12,760 INFO sqlalchemy.engine.Engine [generated in 0.01185s] ('section_name', 'section_name')
[2026-04-18 15:52:12,990] [INFO] Extracted 7 normalized sections.
[2026-04-18 15:52:12,990] [INFO] get_normalized_sections completed in 1.11s


2026-04-18 15:52:13,009 INFO sqlalchemy.engine.Engine ROLLBACK


In [6]:
sections_names

['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [7]:
import json
import numpy as np
import ollama
import asyncio



class QueryClassifier:

    def __init__(self, llm_model="qwen2.5:7b", embed_model="nomic-embed-text", threshold=0.55):
        self.llm_model = llm_model
        self.embed_model = embed_model
        self.threshold = threshold

        self.section_service = SectionService()

        self.section_list = []
        self.section_embeddings = None

    # ----------------------------
    # SINGLE EMBEDDING CALL (OLLAMA)
    # ----------------------------
    async def _embed(self, text: str):

        def _call():
            return ollama.embeddings(
                model=self.embed_model,
                prompt=f"search_document: {text}"
            )

        response = await asyncio.to_thread(_call)
        return response["embedding"]

    # ----------------------------
    # Load + embed DB sections
    # ----------------------------
    async def prepare_context(self):

        self.section_list = await self.section_service.get_normalized_sections()

        if not self.section_list:
            self.section_list = ["default"]

        vectors = []
        for s in self.section_list:
            emb = await self._embed(s)
            vectors.append(emb)

        vectors = np.array(vectors, dtype="float32")

        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
        self.section_embeddings = vectors / norms

    # ----------------------------
    # cosine match helper
    # ----------------------------
    async def _cosine_match(self, text: str):

        vec = await self._embed(text)
        vec = np.array(vec, dtype="float32")
        vec = vec / (np.linalg.norm(vec) + 1e-9)

        sims = np.dot(self.section_embeddings, vec)

        top2_idx = np.argsort(sims)[-2:][::-1]

        results = []
        for idx in top2_idx:
            if sims[idx] >= self.threshold:
                results.append(self.section_list[idx])

        return results[:2] if results else self.section_list[:2]

    # ----------------------------
    # MAIN: batch classification
    # ----------------------------
    async def classify_batch(self, queries: list[str]) -> dict:

        if not self.section_list:
            await self.prepare_context()

        prompt = f"""
You are a classifier.

Available sections:
{self.section_list}

Return JSON like:
{{
  "query": ["section1", "section2"]
}}

Queries:
{queries}
"""

        response = ollama.chat(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0}
        )

        content = response["message"]["content"]

        try:
            raw_output = json.loads(content)

            final_output = {}

            for query, sections in raw_output.items():

                normalized_sections = []

                for s in sections:
                    s = normalize_section(s)
                    matches = await self._cosine_match(s)
                    normalized_sections.extend(matches)

                final_output[query] = list(dict.fromkeys(normalized_sections))[:2]

            return final_output

        except Exception:

            # fallback: embedding-only routing
            return {
                q: await self._cosine_match(q)
                for q in queries
            }

In [8]:
section_dict = await QueryClassifier().classify_batch(query_batch)

2026-04-18 15:52:57,767 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 15:52:57,924 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 15:52:57,941 INFO sqlalchemy.engine.Engine [cached since 45.19s ago] ('section_name', 'section_name')
[2026-04-18 15:52:58,112] [INFO] Extracted 7 normalized sections.
[2026-04-18 15:52:58,112] [INFO] get_normalized_sections completed in 0.66s
2026-04-18 15:52:58,191 INFO sqlalchemy.engine.Engine ROLLBACK


In [9]:
section_dict

{'discuss the terms and conditions for an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'detail the specifics of a vehicle loan product': ['drivesure auto loan dal',
  'edugrow education loan']}

In [10]:
import faiss
import numpy as np
from collections import defaultdict
from sqlalchemy import select
from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("vectorstore")
current_logger.set(logger)


class VectorStore:

    def __init__(self, dim=768):
        self.dim = dim

        # section -> FAISS index
        self.indexes = {}

        # section -> {fid -> ChunkModel}
        self.chunk_store = {}

        self.section_service = SectionService()
        self.valid_sections = set()

    # -------------------------------------------------
    # LOAD CANONICAL SECTIONS
    # -------------------------------------------------
    async def load_sections(self):

        sections = await self.section_service.get_normalized_sections()
        self.valid_sections = set(sections)

        for sec in self.valid_sections:
            if sec not in self.indexes:
                self.indexes[sec] = faiss.IndexHNSWFlat(self.dim, 32)
                self.chunk_store[sec] = {}

    # -------------------------------------------------
    # BUILD VECTOR STORE FROM DB
    # -------------------------------------------------
    @track_performance
    async def add(self):

        try:
            if not self.valid_sections:
                await self.load_sections()

            grouped = defaultdict(list)

            # ---------------- DB FETCH ----------------
            async for session in get_session():

                stmt = select(ChunkModel)
                result = await session.execute(stmt)
                chunks = result.scalars().all()

                for c in chunks:

                    # ❗ SAFE embedding check (FIXED BUG)
                    if c.embedding is None:
                        logger.warning(f"Skipping chunk {c.id} (no embedding)")
                        continue

                    embedding = np.asarray(c.embedding, dtype="float32")

                    if embedding.size == 0:
                        logger.warning(f"Skipping chunk {c.id} (empty embedding)")
                        continue

                    section = normalize_section(
                        c.chunk_metadata.get("section_name")
                    )

                    if section not in self.valid_sections:
                        section = "default"

                    grouped[section].append((c, embedding))

                break

            # -------------------------------------------------
            # BUILD FAISS INDEX PER SECTION
            # -------------------------------------------------
            for section, items in grouped.items():

                index = self.indexes.get(section)

                if index is None:
                    index = faiss.IndexHNSWFlat(self.dim, 32)
                    self.indexes[section] = index
                    self.chunk_store[section] = {}

                vectors = np.array(
                    [emb for _, emb in items],
                    dtype="float32"
                )

                faiss.normalize_L2(vectors)

                start_id = index.ntotal
                index.add(vectors)

                for i, (chunk, _) in enumerate(items):
                    fid = start_id + i
                    self.chunk_store[section][fid] = chunk

            logger.info(
                f"VectorStore built successfully | sections={len(grouped)}"
            )

        except Exception as e:
            logger.error("Failed to build VectorStore from DB")
            raise CustomException(e)

    # -------------------------------------------------
    # SEARCH
    # -------------------------------------------------
    def search(self, query_vector, sections, k=5):

        query = np.asarray(query_vector, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(query)

        results = []

        for sec in sections:

            sec = normalize_section(sec)

            index = self.indexes.get(sec)
            if not index:
                continue

            D, I = index.search(query, k)

            for score, idx in zip(D[0], I[0]):

                if idx == -1:
                    continue

                chunk = self.chunk_store[sec].get(idx)

                if chunk:
                    results.append({
                        "chunk": chunk,
                        "score": float(score)
                    })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]

In [11]:
import asyncio

store = VectorStore(dim=768)

await store.add()

2026-04-18 15:54:04,826 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 15:54:04,837 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 15:54:04,841 INFO sqlalchemy.engine.Engine [cached since 112.1s ago] ('section_name', 'section_name')
[2026-04-18 15:54:04,857] [INFO] Extracted 7 normalized sections.
[2026-04-18 15:54:04,865] [INFO] get_normalized_sections completed in 0.08s
2026-04-18 15:54:04,932 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-18 15:54:05,138 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 15:54:05,174 INFO sqlalchemy.engine.Engine SELECT document_chunks.id, document_chunks.chunk_text, document_chunks.embedding, document_chunks.confidence_score, document_chunks.chunk_metadata, document_chunks.prev_chunk_id, document_chunks.next_chunk_id, document_chunks.created_at, document_chunks.context_chunks 
FROM 

2026-04-18 15:54:06,156 INFO sqlalchemy.engine.Engine ROLLBACK


In [12]:
from src.RAG.models import ChunkModel
from src.RAG.Strategies_RAG_build.embeddings import DocumentEmbedder
import asyncio

async def run_flat_search(store, section_dict, k=10):

    all_results = []

    for query, sections in section_dict.items():

        # 1. embed query
        embedder = DocumentEmbedder()
        query_vector = await embedder.embed_query(query)

        # 2. search FAISS
        results = store.search(
            query_vector=query_vector,
            sections=sections,
            k=k
        )

        # 3. append in required format
        for r in results:
            all_results.append({
                "chunk": r["chunk"],   # ChunkModel
                "score": r["score"]
            })

    # optional: global sort across all queries
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return all_results[:k]

In [13]:
sections_names


['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [14]:
section_dict


{'discuss the terms and conditions for an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'detail the specifics of a vehicle loan product': ['drivesure auto loan dal',
  'edugrow education loan']}

In [15]:
from src.RAG.models import ChunkModel
from src.RAG.Strategies_RAG_build.embeddings import DocumentEmbedder
import asyncio

async def run_flat_search(store, section_dict, k=10):
    all_results = []
    seen_ids = set() # To track unique chunks
    embedder = DocumentEmbedder()

    for query, sections in section_dict.items():
        # 1. Use only the first entry of the section list (0 index)
        target_section = [sections[0]] if sections else []

        # 2. Embed query
        query_vector = await embedder.embed_query(query)

        # 3. Search FAISS
        query_results = store.search(
            query_vector=query_vector,
            sections=target_section,
            k=k
        )

        # 4. Filter and Deduplicate per query
        for r in query_results:
            chunk = r["chunk"]
            # Assuming ChunkModel has an 'id' or 'metadata["id"]' 
            # If no ID exists, use hash(chunk.page_content)
            chunk_id = getattr(chunk, 'id', chunk.id)

            if chunk_id not in seen_ids:
                all_results.append({
                    "chunk": chunk,
                    "score": r["score"]
                })
                seen_ids.add(chunk_id)

    # 5. Global sort and truncate
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return all_results[:k]

In [16]:
results = await run_flat_search(
    store=store,
    section_dict= section_dict,
    k=10
)



In [17]:
results[1]

{'chunk': ChunkModel(chunk_text='. For old vehicles, a margin of 40% of agreed sale price is stipulated', embedding=array([ 4.80474681e-02,  4.28100795e-01, -3.59991050e+00, -6.85792506e-01,
         1.02018678e+00,  7.28988349e-02,  1.48138642e-01,  4.61360902e-01,
         5.71703434e-01,  2.81060904e-01,  6.93947256e-01, -8.13180327e-01,
         4.36914325e-01, -1.88741505e-01,  1.55689156e+00, -3.28988731e-01,
         7.23706931e-02, -7.66723633e-01,  7.06163466e-01,  1.26687670e+00,
        -3.16037059e-01, -7.88400650e-01, -1.80782390e+00, -9.21320915e-01,
         1.33637369e+00, -3.38221490e-01, -5.76804399e-01, -4.04485881e-01,
         5.35466135e-01,  8.94787371e-01,  4.90559191e-01, -2.51092792e-01,
         9.82776880e-01, -1.31240070e+00, -1.50577199e+00, -1.81024051e+00,
         1.08121693e+00, -3.43660749e-02,  6.12758100e-01, -5.42268097e-01,
         8.46621871e-01,  8.45178843e-01, -1.64695129e-01, -1.33035958e-01,
         7.94987679e-01, -5.50287783e-01,  7.6817

In [18]:
for sec, idx in store.indexes.items():
    print(sec, idx.ntotal)

homesure mortgage loan hml 119
edugrow education loan 92
tradeplus business loan tbl 113
homesecure home loan hhl 53
securesave loan against approved securities slaas 73
drivesure auto loan dal 86
flexiserve personal loan fpl 78


In [19]:
import asyncio
from typing import List, Dict, Any
def extract_unique_texts(results_list: List[Dict[str, Any]]) -> List[str]:
        """
        Extracts primary chunk_text and all context_chunks text into a unique list.
        """
        all_texts = set()
        
        for item in results_list:
            chunk_obj = item.get("chunk")
            if not chunk_obj:
                continue

            # 1. Add the primary text
            if hasattr(chunk_obj, 'chunk_text'):
                all_texts.add(chunk_obj.chunk_text)

            # 2. Add text from context_chunks (list of dicts)
            context_chunks = getattr(chunk_obj, 'context_chunks', [])
            for ctx in context_chunks:
                if isinstance(ctx, dict) and 'chunk_text' in ctx:
                    all_texts.add(ctx['chunk_text'])
                elif hasattr(ctx, 'chunk_text'):
                    all_texts.add(ctx.chunk_text)

        unique_list = list(all_texts)
        print(f"Extraction Complete: Found {len(unique_list)} unique text segments.")
        return unique_list

In [20]:
final_results = extract_unique_texts(results_list=results)
final_results

Extraction Complete: Found 40 unique text segments.


['or 85% of the on-road price , whichever is lower.',
 'continuity must be independently assessed',
 'The facility is available to resident individuals including salaried employees, businessmen,',
 'A concession of up to 0.25% is available on car loans where the borrower offers minimum 50% of the',
 'Base Rate + 4.00% \uf0b7 Staff / Ex-staff borrowers under the public scheme are charged at Base Rate',
 '. The product is intended exclusively for non-commercial, private use and provides structured',
 'the purchase of new and pre-owned private vehicles , two-wheelers , and eco-friendly CNG/LPG gas',
 'jeeps, and station wagons for private use; old four-wheelers not more than three years old ; new',
 'The minimum age of the applicant is 21 years and of the co-applicant 18 years',
 '. The vehicle shall be registered only in the name of the primary borrower .',
 'options while ensuring strong asset protection and regulatory compliance.',
 '. For old vehicles, a margin of 40% of agreed sale p

In [21]:
from typing import List, Dict, Any
from sentence_transformers import CrossEncoder

from src.Utils.logger_setup import setup_logger, current_logger, track_performance

logger = setup_logger("reranker")
current_logger.set(logger)


class CrossEncoderReranker:
    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        device: str = "cpu",
        batch_size: int = 8
    ):
        """
        Cross-encoder reranker (sync version, optimized with batching).
        """
        self.model_name = model_name
        self.batch_size = batch_size

        self.model = CrossEncoder(model_name, device=device)

        logger.info(
            f"Initialized CrossEncoderReranker | model={model_name} | device={device} | batch_size={batch_size}"
        )

    # -----------------------------
    # FAST UNIQUE TEXT EXTRACTION
    # -----------------------------
    def _extract_unique_texts(self, results_list: List[Dict[str, Any]]) -> List[str]:
        seen = set()
        flat_texts = []

        for item in results_list:
            chunk = item.get("chunk")
            if not chunk:
                continue

            text = getattr(chunk, "chunk_text", None)
            if text and text not in seen:
                seen.add(text)
                flat_texts.append(text)

            for ctx in getattr(chunk, "context_chunks", []) or []:
                ctx_text = (
                    ctx.get("chunk_text") if isinstance(ctx, dict)
                    else getattr(ctx, "chunk_text", None)
                )

                if ctx_text and ctx_text not in seen:
                    seen.add(ctx_text)
                    flat_texts.append(ctx_text)

        logger.info(f"Extracted {len(flat_texts)} unique texts")
        return flat_texts

    # -----------------------------
    # MAIN RERANK FUNCTION
    # -----------------------------
    @track_performance
    def rerank_chunks(
        self,
        queries: List[str],
        results_list: List[Dict[str, Any]],
        top_n: int = 10
    ) -> Dict[str, List[str]]:

        if not queries or not results_list:
            logger.warning("Empty queries or results_list provided.")
            return {}

        # Step 1: flatten texts
        flat_texts = self._extract_unique_texts(results_list)
        if not flat_texts:
            return {}

        # Step 2: merge query
        merged_query = " ".join(queries)

        # Step 3: create pairs
        pairs = [(merged_query, text) for text in flat_texts]

        # Step 4: inference
        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False
        )

        # Step 5: rank
        ranked = sorted(
            zip(flat_texts, scores),
            key=lambda x: x[1],
            reverse=True
        )

        # Step 6: top N
        top_texts = [text for text, _ in ranked[:top_n]]

        logger.info(
            f"Reranked {len(flat_texts)} texts → returning top {top_n}"
        )

        # ✅ FINAL OUTPUT FORMAT
        return {
            merged_query: top_texts
        }

c:\Users\prasa\anaconda3\envs\olm-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
reranker = CrossEncoderReranker()
top_results = reranker.rerank_chunks(query_batch, results)
print(top_results)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1754.98it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-18 15:56:46,040] [INFO] Initialized CrossEncoderReranker | model=cross-encoder/ms-marco-MiniLM-L-6-v2 | device=cpu | batch_size=8
[2026-04-18 15:56:46,040] [INFO] Extracted 40 unique texts
[2026-04-18 15:56:46,879] [INFO] Reranked 40 texts → returning top 10
[2026-04-18 15:56:46,879] [INFO] rerank_chunks completed in 0.83s
{'discuss the terms and conditions for an automobile financing plan detail the specifics of a vehicle loan product': ['. The product enables borrowers to acquire personal vehicles with flexible repayment options while', 'vehicle financing solution that balances customer affordability , operational discipline , and', 'DriveSure Auto Loan (DAL) is a secured retail lending product designed to finance the purchase of', '. Vehicles financed under this scheme must not be registered or used as commercial vehicles.', 'The loan may be availed for the purchase of new four-wheelers such as cars, jeeps, and station', 'A concession of up to 0.25% is available on car loan

In [23]:
from typing import Dict, List, AsyncGenerator
import asyncio
import ollama


class LLMGenerator:
    def __init__(self, model: str = "phi3.5"):
        self.model = model

    # -----------------------------
    # PROMPT BUILDER
    # -----------------------------
    def _build_prompt(self, query: str, contexts: List[str]) -> str:
        context_block = "\n\n".join(
            [f"[Doc {i+1}]\n{c}" for i, c in enumerate(contexts)]
        )

        return f"""
You are an expert retail banking loan advisor specializing in personal loans, car loans, home loans, interest rates, fees, and repayment terms.

Answer strictly using the provided context only.

If context is insufficient, say: "Not enough information in the provided documents."

Be concise, accurate, and customer-friendly.

User Question:
{query}

Context:
{context_block}

Output:
- Direct Answer
- Key Points
""".strip()

    # -----------------------------
    # ASYNC STREAMING GENERATION
    # -----------------------------
    async def stream_generate(
        self,
        rerank_output: Dict[str, List[str]]
    ) -> AsyncGenerator[Dict[str, str], None]:
        """
        Async generator yielding streaming responses per query.
        """

        loop = asyncio.get_event_loop()

        for query, contexts in rerank_output.items():

            prompt = self._build_prompt(query, contexts)

            print(f"\n\n===== Query: {query} =====\n")

            full_response = ""

            # Run ollama.chat in a thread (since it's blocking)
            stream = await loop.run_in_executor(
                None,
                lambda: ollama.chat(
                    model=self.model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a precise and compliant retail banking loan advisor."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    stream=True
                )
            )

            # Iterate streamed tokens
            for chunk in stream:
                token = chunk["message"]["content"]
                full_response += token

                print(token, end="", flush=True)

                # yield partial updates (optional for UI streaming)
                await asyncio.sleep(0)

            yield {query: full_response}

In [24]:
import asyncio

async def main():
    generator = LLMGenerator(model="qwen2.5:7b")

    rerank_output = top_results  

    async for result in generator.stream_generate(rerank_output):
        print("\n\nFINAL OUTPUT:\n", result)




In [25]:
await main()



===== Query: discuss the terms and conditions for an automobile financing plan detail the specifics of a vehicle loan product =====

Direct Answer:
DriveSure Auto Loan (DAL) offers flexible terms and conditions for financing personal vehicle purchases. 

Key Points:
1. **Eligible Vehicles**: The loan covers new four-wheelers including cars, jeeps, and station wagons.
2. **Registration**: The vehicle must be registered only in the name of the primary borrower.
3. **Commercial Use Restrictions**: Vehicles financed cannot be used for commercial purposes.
4. **Security**: It is a secured retail lending product, meaning the vehicle serves as collateral.
5. **Fees and Charges**:
   - One-time processing charge: 0.50% of the loan amount (maximum ₹10,000)
   - Prepayment penalty: 2% of the remaining loan balance
6. **Discount Offer**: A concession of up to 0.25% is available if the borrower provides a minimum 50% down payment.
7. **Loan Tenure and Repayment Options**: Flexible repayment term